# Components lab 02: sources

Sources schedule emission attempts, create qstate-backed signals, and optionally emit preparation reports. The useful thing to watch is not just the signal: it is the attempt time, qstate ref, subsystem label, and report.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field

import numpy as np

from simyuj.components import Port, PortDelivery, PortDirection, PortKind, connect_ports
from simyuj.components.sources import DeltaTiming, EntangledPairSource, GaussianTiming
from simyuj.components.sources.single_photon_source import SinglePhotonSource
from simyuj.engine import Component, Timeline
from simyuj.qstate import StateSampler
from simyuj.qstate.noise import depolarizing, two_qubit_depolarizing
from simyuj.qstate.state import bell_fidelity, purity
from simyuj.signal import Signal

## 1. Small receivers for the lab

A source talks through ports, so the receivers only need input ports and `handle_event`.

In [ ]:
@dataclass(slots=True)
class QuantumInbox(Component):
    component_id: str
    input_port: Port = field(init=False)
    received: list[tuple[int, Signal, PortDelivery]] = field(default_factory=list)

    def __post_init__(self) -> None:
        self.input_port = Port(
            name='in',
            owner=self,
            owner_id=self.component_id,
            port_kind=PortKind.QUANTUM,
            direction=PortDirection.INGRESS,
        )

    def handle_event(self, event, timeline) -> None:
        delivery = event.payload_ref
        if not isinstance(delivery, PortDelivery):
            raise TypeError('expected PortDelivery')
        signal = delivery.payload
        if not isinstance(signal, Signal):
            raise TypeError('expected Signal')
        self.received.append((timeline.current_time, signal, delivery))


In [ ]:
@dataclass(slots=True)
class ReportInbox(Component):
    component_id: str
    input_port: Port = field(init=False)
    reports: list[tuple[int, object]] = field(default_factory=list)

    def __post_init__(self) -> None:
        self.input_port = Port(
            name='in',
            owner=self,
            owner_id=self.component_id,
            port_kind=PortKind.CLASSICAL,
            direction=PortDirection.INGRESS,
        )

    def handle_event(self, event, timeline) -> None:
        delivery = event.payload_ref
        if not isinstance(delivery, PortDelivery):
            raise TypeError('expected PortDelivery')
        self.reports.append((timeline.current_time, delivery.payload))


## 2. A single-photon source with missed attempts

Use a real emission probability and bounded Gaussian timing. The source declares RNG streams when `schedule_start()` is called.

In [ ]:
timeline = Timeline(master_seed=22)
photon_sink = QuantumInbox('photon.receiver')
report_sink = ReportInbox('source.monitor')

sampler = StateSampler(
    states=('|0>', '|1>', '|+>'),
    probabilities=(0.5, 0.25, 0.25),
    labels=('zero', 'one', 'plus'),
)
source = SinglePhotonSource(
    device_id='alice.sps',
    frequency_hz=1e11,                 # one nominal slot every 10 ticks
    emission_probability=0.65,
    duration_s=70e-12,
    sampler=sampler,
    timing_profile=GaussianTiming(
        mean_emission_delay_ticks=1.5,
        emission_delay_stddev_ticks=0.8,
        max_emission_delay_ticks=4,
    ),
    noise_models=(depolarizing(0.05),),
)

connect_ports(source.output_port, photon_sink.input_port, target_action='receive_photon')
connect_ports(source.report_port, report_sink.input_port, target_action='receive_report')

start_event = source.schedule_start(timeline)
print('start event:', start_event)
print('emission period ticks:', source.emission_period_ticks)
print('stop tick:', source.stop_time)

In [ ]:
timeline.run_until(90)

print('timeline stats:', timeline.stats)
print('photons received:', len(photon_sink.received))
print('reports received:', len(report_sink.reports))
print('qstate records:', timeline.qstate.size())

In [ ]:
print('received photons:', len(photon_sink.received))
for time, signal, delivery in photon_sink.received[:3]:
    print(
        ' t=', time,
        'id=', signal.id,
        'state_ref=', signal.state_ref,
        'targets=', tuple(target.label for target in signal.state_targets),
        'wire=', delivery.connection_id,
    )
if len(photon_sink.received) > 3:
    print('more photon rows:', len(photon_sink.received) - 3)


In [ ]:
print('source-local preparation reports:')
for report in source.reports:
    print(report.report_id, 'attempt', report.attempt_index, 'emission', report.emission_index, 'label', report.sampler_label)

print('\nreports through the classical report port:')
for time, report in report_sink.reports:
    print('t=', time, report.report_id, report.signal_ids)

## 3. Look at the qstate left behind by emitted photons

Source noise converted these emitted states to density form. The signal carries the ref and target label; qstate owns the payload.

In [ ]:
for _, signal, _ in photon_sink.received[:3]:
    record = timeline.qstate.record(signal.state_ref)
    print(signal.id, 'rep=', record.rep, 'subsystems=', tuple(str(s) for s in record.layout.subsystems))
    if hasattr(record.payload, 'rho'):
        print('  density diagonal:', np.round(np.diag(record.payload.rho).real, 3).tolist())
        print('  purity:', round(purity(record.payload), 3))

## 4. An entangled-pair source has two quantum outputs

A successful pair creates one shared two-qubit qstate ref, then sends one member signal out each port.

In [ ]:
timeline = Timeline(master_seed=5)
left_sink = QuantumInbox('left.receiver')
right_sink = QuantumInbox('right.receiver')
pair_reports = ReportInbox('pair.monitor')

pair_source = EntangledPairSource(
    device_id='eps',
    frequency_hz=1e12,
    emission_probability=1.0,
    duration_s=1e-12,
    pair_noise_models=(two_qubit_depolarizing(0.15),),
)

connect_ports(pair_source.left_output_port, left_sink.input_port, target_action='receive_left')
connect_ports(pair_source.right_output_port, right_sink.input_port, target_action='receive_right')
connect_ports(pair_source.report_port, pair_reports.input_port, target_action='receive_pair_report')

pair_source.schedule_start(timeline)
timeline.run_until(5)

print('left signals:', len(left_sink.received))
print('right signals:', len(right_sink.received))
print('pair reports:', len(pair_reports.reports))

In [ ]:
left_signal = left_sink.received[0][1]
right_signal = right_sink.received[0][1]

print('left id:', left_signal.id, 'target:', left_signal.state_targets[0].label)
print('right id:', right_signal.id, 'target:', right_signal.state_targets[0].label)
print('shared state ref:', left_signal.state_ref, right_signal.state_ref)
print('correlation meta:', left_signal.correlation_meta, right_signal.correlation_meta)

record = timeline.qstate.record(left_signal.state_ref)
print('stored rep:', record.rep)
print('stored subsystems:', tuple(str(s) for s in record.layout.subsystems))
print('Bell fidelity after pair noise:', round(bell_fidelity(record.payload, 'phi+'), 3))
print('purity:', round(purity(record.payload), 3))

## Keep this model in your head

A source does four jobs: schedule attempts, decide whether an attempt emits, prepare qstate, and transmit signal/report payloads through ports. Timing and skipped attempts are part of the story.